# Machine Learning Analysis
### DSA 210 — Impact of Global Crises on Financial Markets

This notebook applies four methods to the financial crisis dataset:

1. **Logistic Regression** — predict whether next day's S&P 500 return is positive or negative
2. **Random Forest** — same classification task, more powerful model
3. **K-Means Clustering** — group trading days by market behavior, then analyze regime distributions in ±60-day windows around each crisis
4. **Interrupted Time Series (ITS)** — quantify the immediate level change and post-event slope change in cumulative S&P 500 returns around each crisis, using HAC (Newey-West) standard errors to correct for autocorrelation

In [ ]:
# Import libraries
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import os

from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.cluster import KMeans
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import (
    classification_report, confusion_matrix,
    roc_auc_score, RocCurveDisplay
)

plt.style.use('seaborn-v0_8-whitegrid')
print('Libraries imported.')

In [ ]:
# Load data
save_dir = os.path.expanduser('~/Desktop/data')
returns = pd.read_csv(os.path.join(save_dir, 'returns.csv'), index_col=0, parse_dates=True)
macro   = pd.read_csv(os.path.join(save_dir, 'macro.csv'),   index_col=0, parse_dates=True)

# Crisis dates
CRISES = {
    '9/11 (2001)':             '2001-09-11',
    'Iraq War (2003)':         '2003-03-20',
    'Financial Crisis (2008)': '2008-09-15',
    'COVID-19 (2020)':         '2020-03-11',
    'Russia-Ukraine (2022)':   '2022-02-24',
    'Israel-Hamas (2023)':     '2023-10-07',
}

print(f'Returns shape: {returns.shape}')
print(f'Macro shape:   {macro.shape}')
print(f'Date range:    {returns.index[0].date()} to {returns.index[-1].date()}')

## 1. Feature Engineering

We construct the following features to predict next-day S&P 500 direction:

| Feature | Description |
|---|---|
| `VIX` | Current day's VIX (fear index) |
| `Oil` | Current day's oil return |
| `Gold` | Current day's gold return |
| `SP500_lag1` | Yesterday's S&P 500 return |
| `SP500_ma5` | 5-day moving average of S&P 500 returns |
| `SP500_ma20` | 20-day moving average of S&P 500 returns |
| `Defense` | Defense sector return |
| `Energy` | Energy sector return |
| `Fed_Rate` | Federal funds rate |
| `is_crisis` | 1 if within 30 days of a crisis date |

**Target**: `SP500_up` = 1 if next day's S&P 500 return > 0, else 0

In [ ]:
# Build feature dataframe
df = pd.DataFrame(index=returns.index)

# Raw features from returns
df['VIX']        = returns['VIX']
df['Oil']        = returns['Oil']
df['Gold']       = returns['Gold']
df['Defense']    = returns['Defense']
df['Energy']     = returns['Energy']
df['SP500']      = returns['SP500']

# Lagged and rolling features
df['SP500_lag1'] = returns['SP500'].shift(1)
df['SP500_ma5']  = returns['SP500'].rolling(5).mean()
df['SP500_ma20'] = returns['SP500'].rolling(20).mean()

# Macro feature
df['Fed_Rate']   = macro['Fed_Rate'].reindex(returns.index, method='ffill')

# Crisis indicator: 1 if within 30 days of any crisis
df['is_crisis'] = 0
for crisis_date_str in CRISES.values():
    crisis_date = pd.Timestamp(crisis_date_str)
    mask = (df.index >= crisis_date) & (df.index <= crisis_date + pd.Timedelta(days=30))
    df.loc[mask, 'is_crisis'] = 1

# Target: next day SP500 positive?
df['SP500_next'] = returns['SP500'].shift(-1)
df['SP500_up']   = (df['SP500_next'] > 0).astype(int)

# Drop rows with NaN
df.dropna(inplace=True)

print(f'Dataset shape: {df.shape}')
print(f'Target balance:\n{df["SP500_up"].value_counts()}')

In [ ]:
# Define features and target
FEATURES = ['VIX', 'Oil', 'Gold', 'Defense', 'Energy',
            'SP500_lag1', 'SP500_ma5', 'SP500_ma20', 'Fed_Rate', 'is_crisis']

X = df[FEATURES]
y = df['SP500_up']

# Train/test split — 80% train, 20% test (chronological, no shuffle)
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, shuffle=False
)

# Standardize features (important for Logistic Regression and KNN)
scaler  = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled  = scaler.transform(X_test)

print(f'Train size: {X_train.shape[0]} days')
print(f'Test size:  {X_test.shape[0]} days')
print(f'Train period: {X_train.index[0].date()} to {X_train.index[-1].date()}')
print(f'Test period:  {X_test.index[0].date()} to {X_test.index[-1].date()}')

## 2. Logistic Regression

Logistic Regression is a parametric classification model that estimates the probability that the next day's S&P 500 return is positive, given today's market features.

In [ ]:
# Train Logistic Regression
lr = LogisticRegression(max_iter=1000, random_state=42)
lr.fit(X_train_scaled, y_train)

# Predict
y_pred_lr   = lr.predict(X_test_scaled)
y_proba_lr  = lr.predict_proba(X_test_scaled)[:, 1]

# Evaluate
print('=== Logistic Regression ===')
print(classification_report(y_test, y_pred_lr, target_names=['Down', 'Up']))
print(f'AUC: {roc_auc_score(y_test, y_proba_lr):.4f}')

In [ ]:
# Confusion matrix — Logistic Regression
cm_lr = confusion_matrix(y_test, y_pred_lr)

fig, ax = plt.subplots(figsize=(5, 4))
sns.heatmap(cm_lr, annot=True, fmt='d', cmap='Blues',
            xticklabels=['Down', 'Up'], yticklabels=['Down', 'Up'], ax=ax)
ax.set_xlabel('Predicted')
ax.set_ylabel('Actual')
ax.set_title('Confusion Matrix — Logistic Regression')
plt.tight_layout()
plt.show()

In [ ]:
# Feature coefficients — Logistic Regression
coef_df = pd.DataFrame({
    'Feature': FEATURES,
    'Coefficient': lr.coef_[0]
}).sort_values('Coefficient')

fig, ax = plt.subplots(figsize=(8, 5))
colors = ['#d62728' if c < 0 else '#2196F3' for c in coef_df['Coefficient']]
ax.barh(coef_df['Feature'], coef_df['Coefficient'], color=colors)
ax.axvline(0, color='black', linewidth=0.8)
ax.set_title('Logistic Regression Coefficients\n(Blue = increases P(Up), Red = decreases P(Up))')
ax.set_xlabel('Coefficient')
plt.tight_layout()
plt.show()

## 3. Random Forest

Random Forest is an ensemble method that builds many decision trees and combines their predictions. It handles non-linear relationships and interactions between features automatically.

In [ ]:
# Train Random Forest
rf = RandomForestClassifier(n_estimators=200, max_depth=5,
                             min_samples_leaf=20, random_state=42)
rf.fit(X_train, y_train)  # RF does not need scaling

# Predict
y_pred_rf  = rf.predict(X_test)
y_proba_rf = rf.predict_proba(X_test)[:, 1]

# Evaluate
print('=== Random Forest ===')
print(classification_report(y_test, y_pred_rf, target_names=['Down', 'Up']))
print(f'AUC: {roc_auc_score(y_test, y_proba_rf):.4f}')

In [ ]:
# Confusion matrix — Random Forest
cm_rf = confusion_matrix(y_test, y_pred_rf)

fig, ax = plt.subplots(figsize=(5, 4))
sns.heatmap(cm_rf, annot=True, fmt='d', cmap='Blues',
            xticklabels=['Down', 'Up'], yticklabels=['Down', 'Up'], ax=ax)
ax.set_xlabel('Predicted')
ax.set_ylabel('Actual')
ax.set_title('Confusion Matrix — Random Forest')
plt.tight_layout()
plt.show()

In [ ]:
# Feature importance — Random Forest
importance_df = pd.DataFrame({
    'Feature': FEATURES,
    'Importance': rf.feature_importances_
}).sort_values('Importance', ascending=True)

fig, ax = plt.subplots(figsize=(8, 5))
ax.barh(importance_df['Feature'], importance_df['Importance'], color='#2196F3')
ax.set_title('Random Forest Feature Importance')
ax.set_xlabel('Importance')
plt.tight_layout()
plt.show()

## 4. Model Comparison — ROC Curves

In [ ]:
# ROC curves side by side
fig, ax = plt.subplots(figsize=(7, 5))

RocCurveDisplay.from_predictions(y_test, y_proba_lr,
    name=f'Logistic Regression (AUC={roc_auc_score(y_test, y_proba_lr):.3f})',
    ax=ax, color='#2196F3')
RocCurveDisplay.from_predictions(y_test, y_proba_rf,
    name=f'Random Forest (AUC={roc_auc_score(y_test, y_proba_rf):.3f})',
    ax=ax, color='#FF9800')

ax.plot([0,1],[0,1], 'k--', linewidth=0.8, label='Random baseline')
ax.set_title('ROC Curves: Logistic Regression vs Random Forest')
ax.legend()
plt.tight_layout()
plt.show()

## 5. K-Means Clustering

We cluster trading days based on VIX level and S&P 500 return to identify distinct market regimes. This helps us understand whether crisis periods form a distinct cluster.

In [ ]:
# Prepare data for clustering — use full dataset
cluster_features = ['VIX', 'SP500', 'Oil', 'Gold']
cluster_df = df[cluster_features].dropna()

# Standardize
scaler_c = StandardScaler()
X_cluster = scaler_c.fit_transform(cluster_df)

# Find optimal K using elbow method
inertias = []
K_range = range(2, 10)
for k in K_range:
    km = KMeans(n_clusters=k, random_state=42, n_init=10)
    km.fit(X_cluster)
    inertias.append(km.inertia_)

fig, ax = plt.subplots(figsize=(7, 4))
ax.plot(K_range, inertias, 'o-', color='#2196F3')
ax.set_xlabel('Number of Clusters (K)')
ax.set_ylabel('Within-Cluster Sum of Squares')
ax.set_title('Elbow Method for Optimal K')
plt.tight_layout()
plt.show()

In [ ]:
# Fit K-Means with K=3 (calm / turbulent / crash regimes)
km = KMeans(n_clusters=3, random_state=42, n_init=10)
cluster_df = cluster_df.copy()
cluster_df['Cluster'] = km.fit_predict(X_cluster)

# Label clusters by mean VIX (low VIX = calm, high VIX = crisis)
cluster_vix = cluster_df.groupby('Cluster')['VIX'].mean().sort_values()
label_map = {cluster_vix.index[0]: 'Calm',
             cluster_vix.index[1]: 'Turbulent',
             cluster_vix.index[2]: 'Crisis'}
cluster_df['Regime'] = cluster_df['Cluster'].map(label_map)

# Mark crisis dates
crisis_dates = [pd.Timestamp(d) for d in CRISES.values()]
crisis_points = cluster_df[cluster_df.index.isin(
    [cluster_df.index[cluster_df.index.get_indexer([d], method='nearest')[0]] for d in crisis_dates]
)]

# Plot
colors_map = {'Calm': '#2196F3', 'Turbulent': '#FF9800', 'Crisis': '#d62728'}
fig, ax = plt.subplots(figsize=(10, 6))
for regime, group in cluster_df.groupby('Regime'):
    ax.scatter(group['SP500'], group['VIX'],
               alpha=0.3, s=8, label=regime, color=colors_map[regime])

# Highlight actual crisis dates
ax.scatter(crisis_points['SP500'], crisis_points['VIX'],
           s=120, marker='*', color='black', zorder=5, label='Crisis dates')

for idx, row in crisis_points.iterrows():
    label = [k for k, v in CRISES.items() if pd.Timestamp(v) == idx]
    if label:
        ax.annotate(label[0], (row['SP500'], row['VIX']),
                    textcoords='offset points', xytext=(5, 5), fontsize=7)

ax.set_xlabel('S&P 500 Daily Return (%)')
ax.set_ylabel('VIX')
ax.set_title('K-Means Clustering: Market Regimes (K=3)')
ax.legend()
plt.tight_layout()
plt.show()

In [ ]:
# Cluster statistics
print('Cluster Statistics:')
print(cluster_df.groupby('Regime')[['SP500', 'VIX', 'Oil', 'Gold']].mean().round(3))
print('\nCluster sizes:')
print(cluster_df['Regime'].value_counts())

In [ ]:
# What regime were each crisis date in?
print('Crisis Date Regimes:')
for crisis_name, crisis_date_str in CRISES.items():
    crisis_date = pd.Timestamp(crisis_date_str)
    idx = cluster_df.index.get_indexer([crisis_date], method='nearest')[0]
    row = cluster_df.iloc[idx]
    print(f'  {crisis_name}: {row["Regime"]} (VIX={row["VIX"]:.1f}, SP500={row["SP500"]:.2f}%)')

## 6. Regime Distribution Around Crises (±60 days)

The K-Means "Crisis" cluster contains only one outlier day (likely the April 2020 oil price collapse), making single-day classifications unreliable. To get a more robust picture, we examine the ±60-day window around each crisis and count how many days fall into each regime. The timeline plot then visualizes the actual regime sequence, showing whether crises are preceded by gradually building turbulence, or strike suddenly out of calm conditions.

We use a ±60-day window to remain consistent with the ITS analysis.

In [ ]:
# Regime distribution in ±60 days around each crisis
# This addresses the limitation that the "Crisis" cluster has only 1 day —
# instead of relying on a single point, we count regimes across a 121-day window.
WINDOW = 60

# Use a logical regime order (calm -> turbulent -> crisis) instead of alphabetical
regime_order = ['Calm', 'Turbulent', 'Crisis']
all_regimes = [r for r in regime_order if r in cluster_df['Regime'].unique()]

rows = []
for crisis_name, crisis_date_str in CRISES.items():
    crisis_date = pd.Timestamp(crisis_date_str)

    # Find the closest trading day to the crisis (handles weekends/holidays)
    crisis_idx = cluster_df.index.get_indexer([crisis_date], method='nearest')[0]

    # Clip the window at the dataset boundaries
    start_idx = max(0, crisis_idx - WINDOW)
    end_idx   = min(len(cluster_df), crisis_idx + WINDOW + 1)
    window_df = cluster_df.iloc[start_idx:end_idx]

    # Count how many days in the window fall into each regime
    counts = window_df['Regime'].value_counts()
    row = {'Crisis': crisis_name, 'Date': crisis_date_str}
    for r in all_regimes:
        row[r] = int(counts.get(r, 0))
    row['Total'] = len(window_df)
    rows.append(row)

dist_df = pd.DataFrame(rows).set_index('Crisis')
print('Regime distribution in ±60 days around each crisis:')
print(dist_df.to_string())

In [ ]:
# Timeline plot: visualize the actual regime sequence in the ±60-day window
# Each row is one crisis; vertical dashed line marks the crisis date (offset = 0)
regime_colors = {'Calm': '#2196F3', 'Turbulent': '#FF9800', 'Crisis': '#d62728'}

n = len(CRISES)
fig, axes = plt.subplots(n, 1, figsize=(14, 1.5 * n), sharex=True)
if n == 1:
    axes = [axes]

for ax, (crisis_name, crisis_date_str) in zip(axes, CRISES.items()):
    crisis_date = pd.Timestamp(crisis_date_str)
    crisis_idx = cluster_df.index.get_indexer([crisis_date], method='nearest')[0]
    start_idx = max(0, crisis_idx - WINDOW)
    end_idx   = min(len(cluster_df), crisis_idx + WINDOW + 1)
    window_df = cluster_df.iloc[start_idx:end_idx]

    # x-axis: days relative to the crisis (0 = crisis date)
    offsets = np.arange(start_idx - crisis_idx, end_idx - crisis_idx)

    # Draw a colored band for each day, colored by its regime
    for offset, regime in zip(offsets, window_df['Regime']):
        ax.axvspan(offset - 0.5, offset + 0.5,
                   color=regime_colors[regime], alpha=0.75)

    # Mark the crisis date itself
    ax.axvline(x=0, color='black', linestyle='--', linewidth=1.2)
    ax.set_title(f'{crisis_name} ({crisis_date_str})',
                 loc='left', fontsize=9, fontweight='bold')
    ax.set_yticks([])
    ax.set_xlim(-WINDOW, WINDOW)

# Shared legend at the bottom
handles = [plt.Rectangle((0, 0), 1, 1, color=regime_colors[r], alpha=0.75)
           for r in all_regimes]
fig.legend(handles, all_regimes, loc='lower center',
           ncol=len(all_regimes), bbox_to_anchor=(0.5, -0.02))
axes[-1].set_xlabel('Days from crisis event (0 = crisis date)')
fig.suptitle('Market Regime Timeline Around Each Crisis (±60 days)',
             fontsize=12, y=1.005)
plt.tight_layout()
plt.show()

## Summary of ML Results

### Dataset
- 6,123 trading days (2000–2024), 10 features
- Target: next-day S&P 500 direction (Up/Down)
- Train: 2000–2020 (4,898 days), Test: 2020–2024 (1,225 days)
- Class balance: 3,283 Up vs 2,840 Down — reasonably balanced

### Classification Results
| Model | Accuracy | AUC | Notes |
|---|---|---|---|
| Logistic Regression | 0.54 | 0.516 | Slightly better than random |
| Random Forest | 0.53 | 0.507 | Similar performance |

Both models struggle to predict down days (recall ≈ 0.15–0.16 for Down class), but correctly identify up days well (recall ≈ 0.86–0.87). This reflects the overall upward bias of markets.

### Feature Importance (Random Forest)
- **Defense and Energy** are the most important features — sector behavior carries strong predictive signal
- **SP500_ma5, VIX, SP500_ma20** follow — short-term momentum and fear index matter
- **`is_crisis` has near-zero importance** — this is itself an informative finding: a binary "within 30 days of a known crisis" label does not add predictive power beyond what market-based features (VIX, sector returns, momentum) already capture. Crises manifest *through* volatility and sector signals, not as a separate identifiable signal

### Logistic Regression Coefficients
- **Oil and VIX** have positive coefficients — higher oil returns and VIX slightly increase P(Up)
- **Energy and is_crisis** have the most negative coefficients — crisis periods and energy sector drops are associated with down markets
- **SP500_ma5** negative coefficient suggests mean-reversion: recent strong performance predicts a pullback

### K-Means Clustering (K=3)
| Regime | Size | SP500 mean | VIX mean | Interpretation |
|---|---|---|---|---|
| Calm | 4,383 | +0.518% | −2.946 | Normal bull market days |
| Turbulent | 1,739 | −1.199% | +8.339 | High volatility, negative returns |
| Crisis | 1 | −1.788% | +14.889 | Extreme outlier day (April 2020 oil collapse, Oil = −305%) |

**Note on the "Crisis" cluster**: The third cluster contains a single day — almost certainly April 20, 2020, when WTI crude oil futures briefly traded at negative prices for the first time in history. This is a data outlier rather than a meaningful regime. Because a single trading day rarely captures the full market reaction to a crisis (especially when markets close after major events), we analyze regime distributions in the ±60-day window around each crisis instead (see Regime Distribution analysis above).

### Crisis Date Regimes (on the event date)
- **9/11, Iraq War, Russia-Ukraine, Israel-Hamas** → Calm regime on the actual event date
- **Financial Crisis (2008), COVID-19** → Turbulent regime on the actual event date — consistent with EDA findings

### Regime Distribution Around Crises (±60-day window)

To get a more robust picture of each crisis (and to address the fact that the "Crisis" cluster contains only one outlier day), we examine the 121-day window around each crisis:

| Crisis | Calm | Turbulent | Crisis | % Turbulent |
|---|---|---|---|---|
| 9/11 (2001) | 85 | 36 | 0 | 30% |
| Iraq War (2003) | 89 | 32 | 0 | 26% |
| Financial Crisis (2008) | 69 | 52 | 0 | **43%** |
| COVID-19 (2020) | 82 | 38 | 1 | 31% |
| Russia-Ukraine (2022) | 77 | 44 | 0 | 36% |
| Israel-Hamas (2023) | 92 | 29 | 0 | 24% |

**Key findings from the timeline analysis:**

- **Financial Crisis (2008)** has the highest turbulence concentration (43%), and the timeline shows turbulent days cluster *after* the crisis date — consistent with the negative slope coefficient (β₃ = −0.483) found in ITS. The market continued deteriorating rather than recovering.
- **Russia-Ukraine (2022)** shows 36% turbulent days, but the timeline reveals turbulence was already elevated *before* the invasion. This explains why ITS found no significant effect: markets had already priced in the geopolitical risk, so the event itself was indistinguishable from the baseline.
- **Israel-Hamas (2023)** shows the lowest turbulence (24%), with calm regimes dominating both pre- and post-event — consistent with markets pricing this as a regional rather than global event.
- **The single "Crisis" regime day** (visible only in the COVID-19 timeline, ~25 days after March 11, 2020) corresponds to the April 20, 2020 oil futures collapse — a one-time anomaly rather than a recurring market state.
- **9/11 and Iraq War** show similar moderate turbulence (~28%), with Calm regimes dominating both windows. This supports the EDA finding that geopolitical events have shorter, less persistent market impacts compared to financial/health crises.

### Interpretation
- Neither model achieves meaningful predictive power (AUC ≈ 0.51–0.52), consistent with the Efficient Market Hypothesis — if markets were easily predictable, arbitrage would eliminate the pattern
- The near-zero importance of `is_crisis` reinforces this: even knowing a crisis is near does not give predictive edge, because that information is already absorbed into volatility and sector behavior
- Looking only at the crisis date itself can be misleading — markets often close or react slowly to major events. The ±60-day regime distribution analysis provides a more robust view, showing whether crises emerge from gradually building turbulence or strike out of calm conditions
- Geopolitical crises (9/11, Iraq War, Russia-Ukraine, Israel-Hamas) were classified as Calm on their event date, suggesting initial market reactions were muted — consistent with EDA showing delayed effects
- **Limitation**: Test period (2020–2024) includes COVID-19 and post-pandemic regime, making it unusually volatile compared to training data